# Approach 2: Crop instead of resize

`part2_train.ipynb` (Approach 1) resized full 512x512 slices down to 320x320 to make
CPU training fast enough to complete (~52s/epoch). That worked, but investigating the
resulting precision/recall (Task 4 there: precision 0.56, recall 0.55) raised the
question of whether the resize itself was hurting the model: osteotomy site boxes are
already small (average ~25x27px, smallest ~6x8px at the original 512px), and shrinking
the whole image to 320px shrinks those boxes further to an average of ~16x17px and as
small as ~4x5px - very little pixel detail for the model to learn from.

**The idea**: instead of shrinking the whole image, crop out a smaller window around
where the sites actually are and leave everything else out. If the sites are
reliably positioned within a predictable region, this gives the same 320x320 input
size (same training speed) but at **full native pixel resolution** - no blurring at
all - since the model never has to learn from a downsampled image.

**Validating the idea before building on it.** The box-center coordinates of all 1109
annotated boxes in the dataset were checked: mean x-center 0.504, mean y-center 0.378
(of image width/height), i.e. horizontally centered but vertically biased toward the
upper-middle of the frame (expected, since the mandible arch itself isn't necessarily
centered in how these slices were exported). A search over candidate fixed 320x320 crop
windows found that a window centered at pixel (272, 208) of the original 512x512 image
- i.e. x=[112,432], y=[48,368] - captures boxes overlapping it almost everywhere: after
clipping any box that only partially overlaps the window to the visible portion (rather
than dropping it outright), **all 1109 boxes across all 820 images were preserved**,
with only 3 boxes needing any clipping at all.

## Build the cropped dataset

Each 512x512 image is cropped to the fixed 320x320 window `(112, 48, 432, 368)`
(left, top, right, bottom). Labels are re-expressed relative to the new 320x320 crop:
any box is clipped to the crop boundary (not dropped) if it only partially overlaps.

In [1]:
from pathlib import Path
from PIL import Image

DATASET_DIR = Path("../dataset")
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"
OUT_IMAGES_DIR = DATASET_DIR / "images_cropped"
OUT_LABELS_DIR = DATASET_DIR / "labels_cropped"
OUT_IMAGES_DIR.mkdir(exist_ok=True)
OUT_LABELS_DIR.mkdir(exist_ok=True)

CROP_BOX = (112, 48, 432, 368)  # empirically found: captures ~100% of boxes at native resolution
CROP_SIZE = 320

n_images = n_boxes_total = n_boxes_kept = n_boxes_dropped = n_images_became_empty = 0

for img_path in sorted(IMAGES_DIR.glob("*.jpg")):
    label_path = LABELS_DIR / f"{img_path.stem}.txt"

    cropped = Image.open(img_path).crop(CROP_BOX)
    assert cropped.size == (CROP_SIZE, CROP_SIZE)
    cropped.save(OUT_IMAGES_DIR / img_path.name, quality=95)

    new_lines = []
    had_boxes_before = False
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            had_boxes_before = True
            n_boxes_total += 1
            cls, xc, yc, w, h = line.split()
            xc, yc, w, h = float(xc), float(yc), float(w), float(h)
            x1, y1 = (xc - w / 2) * 512, (yc - h / 2) * 512
            x2, y2 = (xc + w / 2) * 512, (yc + h / 2) * 512

            nx1, ny1 = max(x1, CROP_BOX[0]), max(y1, CROP_BOX[1])
            nx2, ny2 = min(x2, CROP_BOX[2]), min(y2, CROP_BOX[3])
            if nx2 <= nx1 or ny2 <= ny1:
                n_boxes_dropped += 1
                continue

            cx1, cy1 = nx1 - CROP_BOX[0], ny1 - CROP_BOX[1]
            cx2, cy2 = nx2 - CROP_BOX[0], ny2 - CROP_BOX[1]
            new_lines.append(
                f"{cls} {(cx1+cx2)/2/CROP_SIZE:.6f} {(cy1+cy2)/2/CROP_SIZE:.6f} "
                f"{(cx2-cx1)/CROP_SIZE:.6f} {(cy2-cy1)/CROP_SIZE:.6f}"
            )
            n_boxes_kept += 1

    (OUT_LABELS_DIR / f"{img_path.stem}.txt").write_text("\n".join(new_lines))
    if had_boxes_before and not new_lines:
        n_images_became_empty += 1
    n_images += 1

print(f"Processed {n_images} images")
print(f"Boxes: {n_boxes_total} total -> {n_boxes_kept} kept, {n_boxes_dropped} dropped ({n_boxes_dropped/n_boxes_total:.2%})")
print(f"Images that lost all their boxes: {n_images_became_empty}")

Processed 820 images
Boxes: 1109 total -> 1109 kept, 0 dropped (0.00%)
Images that lost all their boxes: 0


## Arrange into YOLO's expected layout

Reuses the exact same patient -> split mapping from `dataset/splits.json` (Approach
1's Task 2 split), so results are directly comparable to Approach 1 - both models are
evaluated on the same held-out patients.

In [2]:
import json
import shutil

split_map = json.load(open(DATASET_DIR / "splits.json"))
YOLO_DIR = DATASET_DIR / "yolo_cropped"

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

counts = {"train": 0, "val": 0, "test": 0}
for img_path in sorted(OUT_IMAGES_DIR.glob("*.jpg")):
    patient_id = img_path.stem.split("_")[0]
    split = split_map[patient_id]
    label_path = OUT_LABELS_DIR / f"{img_path.stem}.txt"
    shutil.copy2(img_path, YOLO_DIR / "images" / split / img_path.name)
    shutil.copy2(label_path, YOLO_DIR / "labels" / split / label_path.name)
    counts[split] += 1

print(f"Copied files into {YOLO_DIR.resolve()}")
print(counts)

Copied files into E:\Bone Union Detection\dataset\yolo_cropped
{'train': 574, 'val': 123, 'test': 123}


In [3]:
data_yaml = f"""path: {YOLO_DIR.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site
"""

data_yaml_path = YOLO_DIR / "data.yaml"
data_yaml_path.write_text(data_yaml)
print(data_yaml_path.read_text())

path: E:/Bone Union Detection/dataset/yolo_cropped
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site



## Train

Same training configuration as Approach 1 (100 epochs budget, patience=20,
batch=32, `cache='ram'`), except `imgsz=320` now performs **no resizing at all**
since the cropped images are already exactly 320x320 - the model trains on full
native-resolution pixels throughout. Same seed (42) for comparability.

In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    patience=20,
    imgsz=320,
    batch=32,
    cache="ram",
    project="../runs",
    name="osteotomy_yolov8n_cropped",
    seed=42,
    plots=False,  # avoid saving mosaics/prediction images that embed dataset content
)

New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'


Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\dataset\yolo_cropped\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=osteotomy_yolov8n_cropped, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 1/2 5.6s/it 1.7s<5.6s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.7s/it 3.4s

                   all        123        180      0.669      0.633      0.644      0.237


Speed: 0.3ms preprocess, 24.4ms inference, 0.0ms loss, 0.9ms postprocess per image


## Task 4 (cropped): Evaluate on the held-out test split

Same test patients as Approach 1 (123 images, 25 patients), for a direct comparison.

In [5]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"Loading best checkpoint from {best_weights}")

test_model = YOLO(str(best_weights))
test_metrics = test_model.val(data=str(data_yaml_path), split="test", imgsz=320, plots=False)

print("\nTest set (123 images, 25 patients) - cropped model:")
print(f"Precision: {test_metrics.box.mp:.3f}")
print(f"Recall:    {test_metrics.box.mr:.3f}")
print(f"mAP50:     {test_metrics.box.map50:.3f}")
print(f"mAP50-95:  {test_metrics.box.map:.3f}")

Loading best checkpoint from E:\Bone Union Detection\notebooks\runs\runs\osteotomy_yolov8n_cropped\weights\best.pt
Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 2.10.7 MB/s, size: 22.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/


val: Scanning E:\Bone Union Detection\dataset\yolo_cropped\labels\test... 29 images, 0 backgrounds, 0 corrupt: 23% ━━╸───────── 29/123 86.6it/s 0.1s<1.1s

val: Scanning E:\Bone Union Detection\dataset\yolo_cropped\labels\test... 63 images, 0 backgrounds, 0 corrupt: 51% ━━━━━━────── 63/123 160.9it/s 0.2s<0.4s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 87% ━━━━━━━━━━── 7/8 2.1it/s 3.1s<0.5s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.3it/s 3.5s

                   all        123        173      0.588      0.527      0.485       0.16


Speed: 0.5ms preprocess, 22.9ms inference, 0.0ms loss, 1.0ms postprocess per image



Test set (123 images, 25 patients) - cropped model:
Precision: 0.588
Recall:    0.527
mAP50:     0.485
mAP50-95:  0.160


## Task 5 (cropped): Qualitative results and per-image analysis

Same methodology as Approach 1: IoU >= 0.5 greedy matching, broken down by
ground-truth box count per image. Overlay images saved to `qualitative_results_cropped/`
(gitignored, not committed - embeds dataset slices).

In [6]:
from PIL import Image, ImageDraw

OUT_DIR = Path("../qualitative_results_cropped")
OUT_DIR.mkdir(exist_ok=True)

test_images = sorted((YOLO_DIR / "images" / "test").glob("*.jpg"))


def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, xc, yc, w, h = [float(p) for p in line.split()]
        boxes.append([
            (xc - w / 2) * img_w, (yc - h / 2) * img_h,
            (xc + w / 2) * img_w, (yc + h / 2) * img_h,
        ])
    return boxes


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


IOU_THRESH = 0.5
per_image_results = []

for img_path in test_images:
    label_path = YOLO_DIR / "labels" / "test" / f"{img_path.stem}.txt"
    w, h = Image.open(img_path).size
    gt_boxes = load_gt_boxes(label_path, w, h)

    pred = test_model.predict(str(img_path), imgsz=320, conf=0.25, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) else []
    pred_confs = pred.boxes.conf.cpu().numpy().tolist() if len(pred.boxes) else []

    matched_gt, matched_pred = set(), set()
    for pi, pb in enumerate(pred_boxes):
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            v = iou(pb, gb)
            if v > best_iou:
                best_iou, best_gi = v, gi
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_gi)
            matched_pred.add(pi)

    n_gt, n_pred, n_tp = len(gt_boxes), len(pred_boxes), len(matched_gt)
    per_image_results.append({
        "path": img_path, "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes, "pred_confs": pred_confs,
        "n_gt": n_gt, "n_pred": n_pred, "n_tp": n_tp,
        "n_fn": n_gt - n_tp, "n_fp": n_pred - len(matched_pred),
        "recall": n_tp / n_gt if n_gt > 0 else None,
    })

by_group = {}
for r in per_image_results:
    key = "1 box" if r["n_gt"] == 1 else ("2+ boxes" if r["n_gt"] >= 2 else "0 boxes")
    by_group.setdefault(key, []).append(r)

print("=== Per-image recall by ground-truth box count (test split, cropped model) ===")
for key, items in sorted(by_group.items()):
    n_images = len(items)
    total_gt = sum(r["n_gt"] for r in items)
    total_tp = sum(r["n_tp"] for r in items)
    perfect = sum(1 for r in items if r["recall"] == 1.0)
    print(f"{key}: {n_images} images, {total_gt} GT boxes, "
          f"box-level recall={total_tp/total_gt:.3f}, "
          f"{perfect}/{n_images} images fully detected ({perfect/n_images:.1%})")

total_fp = sum(r["n_fp"] for r in per_image_results)
total_pred = sum(r["n_pred"] for r in per_image_results)
print(f"\nTotal predicted boxes: {total_pred}, false positives: {total_fp} "
      f"({total_fp/total_pred:.1%} of all predictions)")

perfect_cases = [r for r in per_image_results if r["n_gt"] > 0 and r["recall"] == 1.0 and r["n_fp"] == 0]
miss_cases = sorted([r for r in per_image_results if r["n_fn"] > 0], key=lambda r: -r["n_fn"])
fp_cases = sorted([r for r in per_image_results if r["n_fp"] > 0], key=lambda r: -r["n_fp"])
print(f"\nPerfect-detection images: {len(perfect_cases)}/{len(per_image_results)}")
print(f"Images with >=1 missed box: {len(miss_cases)}/{len(per_image_results)}")
print(f"Images with >=1 false positive: {len(fp_cases)}/{len(per_image_results)}")


def draw_overlay(r, out_path):
    img = Image.open(r["path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    for gb in r["gt_boxes"]:
        draw.rectangle(gb, outline=(0, 255, 0), width=2)
    for pb, conf in zip(r["pred_boxes"], r["pred_confs"]):
        draw.rectangle(pb, outline=(255, 0, 0), width=2)
        draw.text((pb[0], max(0, pb[1] - 10)), f"{conf:.2f}", fill=(255, 0, 0))
    img.save(out_path, quality=95)


for category, cases in [("success", perfect_cases[:3]), ("missed_detection", miss_cases[:3]), ("false_positive", fp_cases[:3])]:
    for i, r in enumerate(cases):
        out_path = OUT_DIR / f"{category}_{i}_{r['path'].stem}.jpg"
        draw_overlay(r, out_path)
        print(f"Saved {category} example: {out_path.name} "
              f"(gt={r['n_gt']}, tp={r['n_tp']}, fn={r['n_fn']}, fp={r['n_fp']})")

=== Per-image recall by ground-truth box count (test split, cropped model) ===
1 box: 87 images, 87 GT boxes, box-level recall=0.552, 48/87 images fully detected (55.2%)
2+ boxes: 36 images, 86 GT boxes, box-level recall=0.616, 13/36 images fully detected (36.1%)

Total predicted boxes: 191, false positives: 90 (47.1% of all predictions)

Perfect-detection images: 36/123
Images with >=1 missed box: 62/123
Images with >=1 false positive: 64/123
Saved success example: success_0_1004_1_75.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_1_1004_1_79.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_2_1004_2_65.jpg (gt=1, tp=1, fn=0, fp=0)
Saved missed_detection example: missed_detection_0_163_1_46.jpg (gt=2, tp=0, fn=2, fp=2)
Saved missed_detection example: missed_detection_1_372_1_142.jpg (gt=2, tp=0, fn=2, fp=0)
Saved missed_detection example: missed_detection_2_524_1_117.jpg (gt=3, tp=1, fn=2, fp=0)
Saved false_positive example: false_positive_0_458_1_157.jpg (gt=1,